# Load dataset 

In [12]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(10000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(1000))

# Tokenize

In [13]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Load model and apply LoRA

In [14]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19504.29it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

# Metrics

In [15]:
from peft import PrefixTuningConfig, get_peft_model, TaskType

prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,   # size of prefix
)

model = get_peft_model(model, prefix_config)

model.print_trainable_parameters()

trainable params: 370,178 || all params: 109,853,956 || trainable%: 0.3370


# Training

In [17]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./weights/prefix_imdb",
    eval_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./weights/logs",
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Train

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.629366,0.529986
2,0.498072,0.397710
3,0.424373,0.383145


TrainOutput(global_step=1875, training_loss=0.49311105143229167, metrics={'train_runtime': 229.7564, 'train_samples_per_second': 130.573, 'train_steps_per_second': 8.161, 'total_flos': 3946736701440000.0, 'train_loss': 0.49311105143229167, 'epoch': 3.0})

In [21]:
model.save_pretrained("./weights/prefix_imdb")
tokenizer.save_pretrained("./weights/prefix_imdb")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]


('./weights/prefix_imdb\\tokenizer_config.json',
 './weights/prefix_imdb\\tokenizer.json')

# Inference

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import torch

base = "bert-base-uncased"
adapter_path = "./weights/prefix_imdb"

tokenizer = AutoTokenizer.from_pretrained(base)

base_model = AutoModelForSequenceClassification.from_pretrained(base, num_labels=2)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

text = "This movie was amazing!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    pred = model(**inputs).logits.argmax(dim=-1).item()

print("positive" if pred == 1 else "negative")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18090.65it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

ValueError: Can't find 'adapter_config.json' at './weights/prefix_imdb'

: 

# Inference 

### Load the saved model 

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import torch

base_model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained("./weights/imdb-lora-adapter")

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2
)

model = PeftModel.from_pretrained(
    base_model,
    "./weights/imdb-lora-adapter"
)

model.eval()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 16650.01it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
      

### Predict sentiment

In [11]:
text = """
This movie was fantastic. The acting was excellent and
the story kept me engaged from beginning to end.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = outputs.logits.argmax(dim=-1).item()

label_map = {
    0: "negative",
    1: "positive"
}

print(label_map[prediction])

positive


### Get probabilities

In [12]:
import torch.nn.functional as F

with torch.no_grad():
    logits = model(**inputs).logits

probs = F.softmax(logits, dim=-1)

print("Negative:", probs[0][0].item())
print("Positive:", probs[0][1].item())

Negative: 0.003719477914273739
Positive: 0.9962804913520813


# Batch inference

In [13]:
texts = [
    "This movie was amazing!",
    "Worst film I have ever seen.",
    "It was okay, not great."
]

inputs = tokenizer(
    texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

with torch.no_grad():
    logits = model(**inputs).logits

preds = logits.argmax(dim=-1)

for text, pred in zip(texts, preds):
    print(text)
    print(label_map[pred.item()])
    print()

This movie was amazing!
positive

Worst film I have ever seen.
negative

It was okay, not great.
negative

